# Auditoría de Videos Publicitarios con Vertex AI y BigQuery
Este notebook captura información de un video, envía el prompt a Vertex AI, recibe la evaluación y guarda los resultados en BigQuery.

In [ ]:
# Instalación de librerías necesarias
!pip install google-cloud-bigquery google-cloud-storage google-cloud-aiplatform ipywidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from google.cloud import bigquery, aiplatform
import datetime

# Formulario de captura de datos
video_url = widgets.Text(description="VIDEO_URL")
project_id = widgets.Text(description="PROJECT_ID")
project_zone = widgets.Text(description="PROJECT_ZONE")
bucket_name = widgets.Text(description="BUCKET_NAME")
api_key = widgets.Password(description="API_KEY")
bq_dataset_name = widgets.Text(description="BQ_DATASET_NAME")
bq_table_name = widgets.Text(description="BQ_TABLE_NAME")
brand_name = widgets.Text(description="BRAND_NAME")
brand_products = widgets.Text(description="BRAND_PRODUCTS")
branded_product_categories = widgets.Text(description="BRANDED_PRODUCT_CATEGORIES")
branded_cta = widgets.Text(description="BRANDED_CALL_TO_ACTIONS")
llm_name = widgets.Text(description="LLM_NAME")
max_output_tokens = widgets.IntText(description="MAX_OUTPUT_TOKENS", value=1024)
temperature = widgets.FloatText(description="TEMPERATURE", value=0.7)
top_p = widgets.FloatText(description="TOP_P", value=0.9)
top_k = widgets.IntText(description="TOP_K", value=40)

form = widgets.VBox([
    video_url, project_id, project_zone, bucket_name, api_key,
    bq_dataset_name, bq_table_name, brand_name, brand_products,
    branded_product_categories, branded_cta, llm_name,
    max_output_tokens, temperature, top_p, top_k
])
display(form)

In [ ]:
# Prompt de auditoría (pegamos aquí el texto completo definido anteriormente)
prompt = """
Actúa como un Auditor de creatividad, efectos visuales y branding, ...
(Aquí va el prompt completo con los 21 criterios y definiciones)
"""

In [ ]:
# Llamada a Vertex AI
def call_vertex(video_url_value, brand_name_value):
    aiplatform.init(project=project_id.value, location=project_zone.value)
    model = aiplatform.TextGenerationModel.from_pretrained(llm_name.value)
    response = model.predict(
        prompt + f"\n\nVideo URL: {video_url_value}\nBrand: {brand_name_value}",
        temperature=temperature.value,
        max_output_tokens=max_output_tokens.value,
        top_p=top_p.value,
        top_k=top_k.value
    )
    return response.text

In [ ]:
# Transformación de la respuesta a formato BigQuery
def transform_to_bq(response_text):
    rows_to_insert = [
        {
            "execution_time": datetime.datetime.now().isoformat(),
            "video_id": video_url.value.split("v=")[-1],
            "video_name": "Video Evaluado",
            "brand_name": brand_name.value,
            "evaluation_result": response_text
        }
    ]
    return rows_to_insert

In [ ]:
# Inserción en BigQuery
def insert_into_bigquery(rows):
    client = bigquery.Client(project=project_id.value)
    table_id = f"{project_id.value}.{bq_dataset_name.value}.{bq_table_name.value}"
    errors = client.insert_rows_json(table_id, rows)
    if errors == []:
        print("Datos insertados correctamente en BigQuery.")
    else:
        print("Errores al insertar:", errors)

In [ ]:
# Flujo completo
def run_pipeline():
    response = call_vertex(video_url.value, brand_name.value)
    rows = transform_to_bq(response)
    insert_into_bigquery(rows)

run_button = widgets.Button(description="Ejecutar Evaluación")
display(run_button)

def on_button_click(b):
    run_pipeline()

run_button.on_click(on_button_click)